# 02 Exploratory Data Analysis

This notebook explores employee attrition patterns and translates them into HR business questions around workload, satisfaction, tenure, promotion, and evaluation practices.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 50)

In [ ]:
from feature_engineering import add_retention_features
from utils import load_processed_data, save_plot

clean_df = load_processed_data()
df = add_retention_features(clean_df)
df.head()

## Class Distribution

The target variable is imbalanced: most employees stayed, while a smaller but business-critical group left. This means recall and precision are more informative than accuracy alone.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x="left", ax=ax)
ax.set_title("Employee Attrition Class Distribution")
ax.set_xlabel("Left company (0 = stayed, 1 = left)")
ax.set_ylabel("Employee count")
save_plot(fig, "images/eda/class_distribution.png")
plt.show()

attrition_rate = df["left"].mean()
print(f"Attrition rate: {attrition_rate:.1%}")

## Correlation Heatmap

The heatmap gives a quick view of numeric relationships. It is especially useful for detecting potential multicollinearity and variables strongly associated with attrition.

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df[numeric_cols].corr(), cmap="vlag", center=0, annot=True, fmt=".2f", ax=ax)
ax.set_title("Correlation Heatmap")
save_plot(fig, "images/eda/correlation_heatmap.png")
plt.show()

## Workload and Project Load

Employees with very high project counts and high monthly hours show elevated attrition risk. This does not prove causality, but it gives HR a concrete operational area to investigate.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x="number_project", y="average_monthly_hours", hue="left", ax=ax)
ax.set_title("Project Load vs. Monthly Hours by Attrition")
ax.set_xlabel("Number of projects")
ax.set_ylabel("Average monthly hours")
ax.legend(title="Left")
save_plot(fig, "images/eda/project_load_vs_hours.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=df, x="average_monthly_hours", y="last_evaluation", hue="left", alpha=0.45, ax=ax)
ax.set_title("Monthly Hours vs. Last Evaluation")
ax.set_xlabel("Average monthly hours")
ax.set_ylabel("Last evaluation score")
save_plot(fig, "images/eda/hours_vs_evaluation.png")
plt.show()

## Satisfaction and Tenure

Satisfaction is strongly related to attrition, but it may be a late-stage signal. For a realistic early-warning model, satisfaction should be interpreted carefully and tested as a potential leakage feature.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x="tenure", y="satisfaction_level", hue="left", ax=ax)
ax.set_title("Satisfaction by Tenure and Attrition")
ax.set_xlabel("Years at company")
ax.set_ylabel("Satisfaction level")
ax.legend(title="Left")
save_plot(fig, "images/eda/satisfaction_by_tenure.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=df, x="average_monthly_hours", y="satisfaction_level", hue="left", alpha=0.45, ax=ax)
ax.set_title("Satisfaction vs. Monthly Hours")
ax.set_xlabel("Average monthly hours")
ax.set_ylabel("Satisfaction level")
save_plot(fig, "images/eda/satisfaction_vs_monthly_hours.png")
plt.show()

## EDA Insights

Key observations from the exploratory analysis:

- Attrition is concentrated in identifiable employee profiles rather than spread randomly.
- High project load and high monthly hours appear repeatedly in groups that left.
- Four-year tenure employees show notable satisfaction and attrition patterns, suggesting a career-stage retention issue.
- Promotion history and salary bands should be interpreted as HR policy signals, not only as model inputs.
- Satisfaction is valuable for explanation, but it may be less realistic for early intervention if collected close to resignation.